In [1]:
list.of.packages <- c("tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=TRUE)  

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transp

# Longer dataframe

In [2]:
df_deg <- fread("./interm/df_T_cells_CDC_longer.csv")
sprintf("%i x %i dataframe", nrow(df_deg), ncol(df_deg))
head(df_deg,2)

[1] "53234477 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.9,27.6,T0DayMin1,2,0,1
42667,96.7,28.3,T0DayMin1,2,0,1


# Categorize T per hour into bins

In [3]:
data.frame(df_deg) %>% 
        select(Temp)%>%
        stargazer(nobs=TRUE, digits = 2,
                  summary.stat = c("n","mean", "sd","min","p25","median","p75","max"),
                  title = c("Summary statistics"),
                  type="text"
                     )


Summary statistics
Statistic     N      Mean  St. Dev. Min Pctl(25) Median Pctl(75) Max
--------------------------------------------------------------------
Temp      53,234,477 22.90   7.96   -27    18      24      28    45 
--------------------------------------------------------------------


In [4]:
thresholds_T <- c(-28,17,21,23,25,27,29,31,33,45)

In [5]:
df_deg_bins <- df_deg %>%
                  mutate(T_bin = cut(Temp, breaks = thresholds_T, 
                                     labels = c("T_Bin_inf17","T_Bin_17_21",
                                                "T_Bin_21_23","T_Bin_23_25","T_Bin_25_27",
                                                "T_Bin_27_29","T_Bin_29_31","T_Bin_31_33",
                                                "T_Bin_sup33"),
                                     include.lowest = TRUE))
sprintf("%i x %i dataframe", nrow(df_deg_bins), ncol(df_deg_bins))
head(df_deg_bins,2)

[1] "53234477 x 8 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min,T_bin
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>,<fct>
42474,91.9,27.6,T0DayMin1,2,0,1,T_Bin_inf17
42667,96.7,28.3,T0DayMin1,2,0,1,T_Bin_inf17


# Pivot wider

## Mean temperature

In [6]:
df_deg_wide <- df_deg_bins %>%
                        select(Measured_date_CDC,cell_x,cell_y,Day_Min,T_bin,N_hours)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y,Day_Min,T_bin)%>%
                        summarize(N_hours=sum(N_hours))%>%
                        ungroup()%>%
                        pivot_wider(names_from = T_bin, values_from = N_hours)%>%
                        ungroup()%>%
                        mutate_all(~replace_na(., 0))
sprintf("%i x %i dataframe", nrow(df_deg_wide), ncol(df_deg_wide))
head(df_deg_wide,2)

`summarise()` has grouped output by 'Measured_date_CDC', 'cell_x', 'cell_y', 'Day_Min'. You can override using the `.groups` argument.


[1] "5209710 x 13 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,T_Bin_17_21,T_Bin_21_23,T_Bin_23_25,T_Bin_25_27,T_Bin_27_29,T_Bin_29_31,T_Bin_inf17,T_Bin_31_33,T_Bin_sup33
<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
42022,74,15.4,1,9,2,3,2,3,5,0,0,0
42022,74,15.4,2,9,2,4,2,2,5,0,0,0


In [7]:
rm(df_deg,df_deg_bins)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1442547,77.1,80133450,4279.6,100166812,5349.5
Vcells,41622581,317.6,741783090,5659.4,926776111,7070.8


In [8]:
df_deg_Nhours_30 <- df_deg_wide %>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Nhours_on30 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_Nhours_30), ncol(df_deg_Nhours_30))
head(df_deg_Nhours_30,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_Bin_17_21_Nhours_on30,T_Bin_21_23_Nhours_on30,T_Bin_23_25_Nhours_on30,T_Bin_25_27_Nhours_on30,T_Bin_27_29_Nhours_on30,T_Bin_29_31_Nhours_on30,T_Bin_inf17_Nhours_on30,T_Bin_31_33_Nhours_on30,T_Bin_sup33_Nhours_on30
<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
42022,74.0,15.4,239,113,90,78,72,93,30,5,0
42022,74.1,15.2,234,113,91,82,71,106,15,8,0


In [9]:
df_deg_Nhours_20 <- df_deg_wide %>%
                        filter(Day_Min <= 20)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Nhours_on20 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_Nhours_20), ncol(df_deg_Nhours_20))
head(df_deg_Nhours_20,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_Bin_17_21_Nhours_on20,T_Bin_21_23_Nhours_on20,T_Bin_23_25_Nhours_on20,T_Bin_25_27_Nhours_on20,T_Bin_27_29_Nhours_on20,T_Bin_29_31_Nhours_on20,T_Bin_inf17_Nhours_on20,T_Bin_31_33_Nhours_on20,T_Bin_sup33_Nhours_on20
<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
42022,74.0,15.4,157,73,63,50,45,57,30,5,0
42022,74.1,15.2,157,77,58,60,44,61,15,8,0


In [10]:
df_deg_Nhours_10 <- df_deg_wide %>%
                        filter(Day_Min <= 10)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Nhours_on10 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_Nhours_10), ncol(df_deg_Nhours_10))
head(df_deg_Nhours_10,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_Bin_17_21_Nhours_on10,T_Bin_21_23_Nhours_on10,T_Bin_23_25_Nhours_on10,T_Bin_25_27_Nhours_on10,T_Bin_27_29_Nhours_on10,T_Bin_29_31_Nhours_on10,T_Bin_inf17_Nhours_on10,T_Bin_31_33_Nhours_on10,T_Bin_sup33_Nhours_on10
<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
42022,74.0,15.4,72,24,23,21,22,43,30,5,0
42022,74.1,15.2,81,23,23,25,21,44,15,8,0


# Merge

In [11]:
result <- data.frame(df_deg_Nhours_30) %>%
                    left_join(df_deg_Nhours_20, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_Nhours_10, by = c("Measured_date_CDC","cell_x","cell_y"))
sprintf("%i x %i dataframe", nrow(result), ncol(result))
head(result,2)

[1] "173657 x 30 dataframe"

,Measured_date_CDC,cell_x,cell_y,T_Bin_17_21_Nhours_on30,T_Bin_21_23_Nhours_on30,T_Bin_23_25_Nhours_on30,T_Bin_25_27_Nhours_on30,T_Bin_27_29_Nhours_on30,T_Bin_29_31_Nhours_on30,T_Bin_inf17_Nhours_on30,T_Bin_31_33_Nhours_on30,T_Bin_sup33_Nhours_on30,T_Bin_17_21_Nhours_on20,T_Bin_21_23_Nhours_on20,T_Bin_23_25_Nhours_on20,T_Bin_25_27_Nhours_on20,T_Bin_27_29_Nhours_on20,T_Bin_29_31_Nhours_on20,T_Bin_inf17_Nhours_on20,T_Bin_31_33_Nhours_on20,T_Bin_sup33_Nhours_on20,T_Bin_17_21_Nhours_on10,T_Bin_21_23_Nhours_on10,T_Bin_23_25_Nhours_on10,T_Bin_25_27_Nhours_on10,T_Bin_27_29_Nhours_on10,T_Bin_29_31_Nhours_on10,T_Bin_inf17_Nhours_on10,T_Bin_31_33_Nhours_on10,T_Bin_sup33_Nhours_on10
,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,42022,74.0,15.4,239,113,90,78,72,93,30,5,0,157,73,63,50,45,57,30,5,0,72,24,23,21,22,43,30,5,0
2,42022,74.1,15.2,234,113,91,82,71,106,15,8,0,157,77,58,60,44,61,15,8,0,81,23,23,25,21,44,15,8,0


# Save

In [12]:
fwrite(result,"./output/df_T_cells_CDC_Nhours.csv")